# 实验 01 · 有限拓扑与连续映射

**对应：第一、二章。** 本 Notebook 区分理论与可运行检查。

理论：拓扑要求任意并、有限交封闭；有限集合只有有限多个子集，因此枚举二元并与二元交即可核验全部条件。有限空间紧致的证明见第二章。

可运行任务：验证两点集合上的三个拓扑，找出一个失败的集合族，再检验连续双射不一定是同胚（homeomorphism）。仅依赖 Python 标准库，无网络与随机输入。


In [1]:
from itertools import combinations

def powerset(points):
    values = sorted(points)
    return {frozenset(c) for k in range(len(values) + 1)
            for c in combinations(values, k)}

def is_topology(points, opens):
    points = frozenset(points)
    opens = set(opens)
    if frozenset() not in opens or points not in opens:
        return False
    if any(not u <= points for u in opens):
        return False
    return all(a | b in opens and a & b in opens
               for a in opens for b in opens)

X = frozenset({0, 1})
discrete = powerset(X)
indiscrete = {frozenset(), X}
sierpinski = {frozenset(), frozenset({1}), X}
for name, topology in [('discrete', discrete), ('indiscrete', indiscrete),
                       ('Sierpinski', sierpinski)]:
    assert is_topology(X, topology)
    print(name, 'is a topology:', True)

Y = frozenset({'a', 'b', 'c'})
bad = {frozenset(), Y, frozenset({'a'}), frozenset({'b'})}
assert not is_topology(Y, bad)
assert frozenset({'a', 'b'}) not in bad
print('Incomplete family is a topology:', is_topology(Y, bad))


discrete is a topology: True
indiscrete is a topology: True
Sierpinski is a topology: True
Incomplete family is a topology: False


## 连续性的可计算条件

给出域的开集族和陪域的开集族，对每个陪域开集 $V$ 计算 $f^{-1}(V)$，再检查其是否为域的开集。这里的逆像并不需要反函数存在。


In [2]:
def is_continuous(domain, codomain, domain_opens, codomain_opens, mapping):
    assert is_topology(domain, domain_opens)
    assert is_topology(codomain, codomain_opens)
    assert set(mapping) == set(domain)
    assert set(mapping.values()) <= set(codomain)
    for target_open in codomain_opens:
        preimage = frozenset(x for x in domain if mapping[x] in target_open)
        if preimage not in domain_opens:
            return False
    return True

identity = {0: 0, 1: 1}
forward = is_continuous(X, X, discrete, indiscrete, identity)
backward = is_continuous(X, X, indiscrete, discrete, identity)
assert len(set(identity.values())) == len(X)
assert forward is True and backward is False
print('Discrete -> indiscrete, continuous:', forward)
print('Inverse, continuous:', backward)
print('Homeomorphism:', forward and backward)

# Independent positive example: constant maps are continuous.
constant = {0: 1, 1: 1}
assert is_continuous(X, X, indiscrete, discrete, constant)
print('Constant map is continuous:', True)


Discrete -> indiscrete, continuous: True
Inverse, continuous: False
Homeomorphism: False
Constant map is continuous: True


## 读懂输出与继续探索

恒等对应是双射，从离散空间到平凡空间连续，但反向不连续。因此“连续双射必是同胚”的断言失败。常值映射则给出正例。

练习：把两个拓扑都换成离散拓扑；再检验谢尔宾斯基空间上交换 0、1 的双射是否连续。修改之前先写出预测。

这些断言只核验指定有限空间，不证明关于任意空间的一般命题。理论证明见教材第二章。

参考：James R. Munkres, *Topology*, 2nd ed., 2000。说明与代码为本项目原创。
